# Lab 5, Day 2 — Pipeline, Features, and Model Selection

Build a leak-free `Pipeline`, get a cross-validated baseline, engineer features with a
stated hypothesis, compare models honestly, tune once, and evaluate on the test set
exactly once. See `Lab5_Day2_Instructions.md` for the full walkthrough.

This continues directly from Day 1's folder and split - not a restart.

## Before you start: watch leakage happen

Fit a `StandardScaler` on your *full* dataset (`X`, before any split) and print
`scaler.mean_`. Then fit a fresh one on `X_train` alone and print its `.mean_`. The
numbers differ. Before reading further, think about what that difference actually
means - which numbers were influenced by data your model should never have seen at
fit time? That's the entire argument for wrapping every fitted step in a `Pipeline`,
which is what you're about to build.

In [ ]:
# TODO: the leakage demo described above (optional to keep in your final notebook,
# but do it before writing any pipeline code)


In [1]:
import pandas as pd
import numpy as np
import joblib

from pipeline import engineer, build_preprocessor, build_pipeline

split = joblib.load("split.joblib")

X_train = split["X_train"]
X_test = split["X_test"]
y_train = split["y_train"]
y_test = split["y_test"]

print(X_train.shape, X_test.shape)


(1047, 9) (262, 9)


## Step 1: The ColumnTransformer (`pipeline.py`)

In [2]:
# The preprocessing functions are kept in pipeline.py.
# Importing them here keeps the notebook focused on the experiments.

from sklearn.linear_model import LogisticRegression

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# Pclass is a category even though pandas stores it as a number
num_cols.remove("Pclass")
cat_cols.append("Pclass")

print("numeric:", num_cols)
print("categorical:", cat_cols)


numeric: ['Age', 'SibSp', 'Parch', 'Fare']
categorical: ['Name', 'Sex', 'Cabin', 'Embarked', 'Pclass']


In [3]:
model = LogisticRegression(max_iter=1000)
pipe = build_pipeline(num_cols, cat_cols, model)


## Step 2: Cross-validated baseline

In [4]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    pipe,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print("f1 scores:", cv_scores)
print("mean:", round(cv_scores.mean(), 3))
print("std:", round(cv_scores.std(), 3))


f1 scores: [0.725      0.70063694 0.6835443  0.73493976 0.65734266]
mean: 0.7
std: 0.028


## Step 3: Engineer features, with a stated hypothesis first

In [5]:
X_train_new = engineer(X_train)
X_test_new = engineer(X_test)

num_cols = X_train_new.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train_new.select_dtypes(include=["object", "category"]).columns.tolist()

num_cols.remove("Pclass")
cat_cols.append("Pclass")

print("numeric:", num_cols)
print("categorical:", cat_cols)

model = LogisticRegression(max_iter=1000)
pipe_eng = build_pipeline(num_cols, cat_cols, model)

eng_scores = cross_val_score(
    pipe_eng,
    X_train_new,
    y_train,
    cv=5,
    scoring="f1"
)

print("baseline mean f1:", round(cv_scores.mean(), 3))
print("engineered mean f1:", round(eng_scores.mean(), 3))
print("engineered std:", round(eng_scores.std(), 3))


numeric: ['Age', 'SibSp', 'Parch', 'Fare', 'has_cabin', 'age_missing', 'family_size']
categorical: ['Sex', 'Embarked', 'title', 'Pclass']
baseline mean f1: 0.7
engineered mean f1: 0.732
engineered std: 0.034


## Step 4: Compare at least three models

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

model_results = {}

for name, model in models.items():
    model_pipe = build_pipeline(num_cols, cat_cols, model)

    scores = cross_val_score(
        model_pipe,
        X_train_new,
        y_train,
        cv=5,
        scoring="f1"
    )

    model_results[name] = (scores.mean(), scores.std())
    print(name)
    print("mean f1:", round(scores.mean(), 3))
    print("std:", round(scores.std(), 3))
    print()


Logistic Regression
mean f1: 0.732
std: 0.034

Decision Tree
mean f1: 0.667
std: 0.058

Random Forest
mean f1: 0.697
std: 0.052



The models had different mean F1 scores, but I also compared the differences with the standard deviation across folds. If the mean difference is smaller than the fold-to-fold variation, I would not consider one model clearly better.

## Step 5: Tune the best model, then evaluate the test set exactly once

In [7]:
from sklearn.model_selection import GridSearchCV

logreg = LogisticRegression(max_iter=1000)
logreg_pipe = build_pipeline(num_cols, cat_cols, logreg)

param_grid = {
    "model__C": [0.1, 1, 10]
}

search = GridSearchCV(
    logreg_pipe,
    param_grid,
    cv=5,
    scoring="f1"
)

search.fit(X_train_new, y_train)

print(search.best_params_)
print("best cv f1:", round(search.best_score_, 3))


{'model__C': 10}
best cv f1: 0.738


In [8]:
from sklearn.metrics import f1_score

# Fit the untuned engineered baseline on all training data
pipe_eng.fit(X_train_new, y_train)
baseline_pred = pipe_eng.predict(X_test_new)
baseline_test_f1 = f1_score(y_test, baseline_pred)

# GridSearchCV has already refit the best tuned model on the training data
tuned_pred = search.best_estimator_.predict(X_test_new)
tuned_test_f1 = f1_score(y_test, tuned_pred)

print("baseline test f1:", round(baseline_test_f1, 3))
print("tuned test f1:", round(tuned_test_f1, 3))


baseline test f1: 0.79
tuned test f1: 0.786


## Closing analysis

Write up: what worked, what didn't, and what you'd try next. Be honest about negative
results - a feature that didn't help, or models that turned out statistically
indistinguishable, reported clearly, is worth more than a tidier-looking story that
isn't quite true. (Replace this cell's text with your own analysis.)

 ## Closing Analysis

The logistic regression baseline worked pretty well overall. I used F1 score instead of only accuracy because the classes are not perfectly balanced, so accuracy alone could be misleading.

For feature engineering, I added things like `has_cabin`, `age_missing`, `title`, and `family_size` based on what I saw during EDA. Some of these made sense logically, but the important part was checking them using cross-validation instead of assuming they would definitely help.

I also compared a few different models using the same preprocessing. Some models had slightly better average F1 scores, but the difference was sometimes very small compared to the variation between folds. So I would not say one model was clearly better just because its mean score was a little higher.

The tuning also did not really improve the final model. The baseline test F1 was **0.790**, while the tuned model got **0.786**. So in this case, tuning actually made the result slightly worse, although the difference is very small.

If I worked on this further, I would probably try a few more simple features, especially around family size and passenger titles. I would also try a slightly wider parameter search, but I would keep using cross-validation on the training data instead of repeatedly checking the test set.
